In [1]:
import sqlite3
import pandas as pd

In [2]:
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

In [3]:
cursor.executescript('''
CREATE TABLE clients (
    client_id INTEGER PRIMARY KEY AUTOINCREMENT,
    name TEXT NOT NULL,
    age INTEGER CHECK (age >= 0),
    registration_date DATE NOT NULL
);

CREATE TABLE accounts (
    account_id INTEGER PRIMARY KEY AUTOINCREMENT,
    client_id INTEGER REFERENCES clients(client_id) ON DELETE CASCADE,
    balance DECIMAL(15,2) NOT NULL CHECK (balance >= 0),
    open_date DATE NOT NULL
);

CREATE TABLE transactions (
    transaction_id INTEGER PRIMARY KEY AUTOINCREMENT,
    account_id INTEGER REFERENCES accounts(account_id) ON DELETE CASCADE,
    amount DECIMAL(15,2) NOT NULL,
    transaction_date DATE NOT NULL,
    transaction_type TEXT NOT NULL CHECK (transaction_type IN ('deposit', 'withdrawal'))
);
''')

In [4]:
cursor.executescript('''
INSERT INTO clients (name, age, registration_date) VALUES
('Иван Иванов', 30, '2019-05-15'),
('Мария Петрова', 25, '2020-01-10'),
('Алексей Сидоров', 40, '2021-03-22'),
('Елена Кузнецова', 35, '2020-07-19'),
('Дмитрий Смирнов', 28, '2022-11-05'),
('Ольга Васнецова', 50, '2018-12-30'),
('Сергей Козлов', 33, '2020-06-14'),
('Анна Морозова', 29, '2021-09-01'),
('Павел Новиков', 45, '2019-08-25'),
('Татьяна Павлова', 31, '2020-04-17');

INSERT INTO accounts (client_id, balance, open_date) VALUES
(1, 15000.00, '2019-05-20'),
(1, 5000.00, '2020-02-10'),
(2, 20000.00, '2020-01-15'),
(3, 30000.00, '2021-03-25'),
(4, 10000.00, '2020-07-25'),
(5, 25000.00, '2022-11-10'),
(6, 40000.00, '2019-01-05'),
(7, 12000.00, '2020-06-20'),
(8, 18000.00, '2021-09-05'),
(9, 22000.00, '2019-09-01'),
(10, 15000.00, '2020-04-20');

INSERT INTO transactions (account_id, amount, transaction_date, transaction_type) VALUES
(1, 1000.00, '2023-01-05', 'deposit'),
(1, 500.00, '2023-01-10', 'withdrawal'),
(2, 2000.00, '2023-02-15', 'deposit'),
(2, 1000.00, '2023-02-20', 'withdrawal'),
(3, 3000.00, '2023-03-25', 'deposit'),
(3, 1500.00, '2023-03-30', 'withdrawal'),
(4, 4000.00, '2023-04-05', 'deposit'),
(4, 2000.00, '2023-04-10', 'withdrawal'),
(5, 5000.00, '2023-05-15', 'deposit'),
(5, 2500.00, '2023-05-20', 'withdrawal'),
(6, 6000.00, '2023-06-25', 'deposit'),
(6, 3000.00, '2023-06-30', 'withdrawal'),
(7, 7000.00, '2023-07-05', 'deposit'),
(7, 3500.00, '2023-07-10', 'withdrawal'),
(8, 8000.00, '2023-08-15', 'deposit'),
(8, 4000.00, '2023-08-20', 'withdrawal'),
(9, 9000.00, '2023-09-25', 'deposit'),
(9, 4500.00, '2023-09-30', 'withdrawal'),
(10, 10000.00, '2023-10-05', 'deposit'),
(10, 5000.00, '2023-10-10', 'withdrawal');
''')

In [5]:
conn.commit()

In [7]:
original_query = '''
SELECT c.client_id, c.name, c.age,
    (SELECT COUNT(*) FROM accounts a WHERE a.client_id = c.client_id) AS total_accounts,
    (SELECT SUM(a.balance) FROM accounts a WHERE a.client_id = c.client_id) AS total_balance,
    (SELECT COUNT(*) FROM transactions t JOIN accounts a ON t.account_id = a.account_id WHERE a.client_id = c.client_id AND t.transaction_type = 'deposit') AS total_deposits,
    (SELECT COUNT(*) FROM transactions t JOIN accounts a ON t.account_id = a.account_id WHERE a.client_id = c.client_id AND t.transaction_type = 'withdrawal') AS total_withdrawals
FROM clients c
WHERE c.registration_date >= '2020-01-01'
ORDER BY total_balance DESC;
'''

## Краткие пояснения

### Что сделано
1) Агрегация данных по счетам и транзакциям вынесена в CTE
2) Использована одна агрегация для подсчёта сумм и количества
3) Добавлена COALESCE на случай, если найдется клиент без счета или операций

### Результат

1) Таблицы `accounts` и `transactions` сканируются только один раз вместо двух для каждого клиента, что улучшает производительность
2) Запрос легче читать, так как вместо подзапросов используются табличные выражения
3) Результаты старого и нового запросов совпадают

In [12]:
optimized_query = '''
WITH client_accounts AS (
    SELECT
        a.client_id,
        COUNT(a.account_id) AS total_accounts,
        SUM(a.balance) AS total_balance
    FROM accounts a
    GROUP BY a.client_id
),
client_transactions AS (
    SELECT
        a.client_id,
        SUM(CASE WHEN t.transaction_type = 'deposit' THEN 1 ELSE 0 END) AS total_deposits,
        SUM(CASE WHEN t.transaction_type = 'withdrawal' THEN 1 ELSE 0 END) AS total_withdrawals
    FROM accounts a
    JOIN transactions t ON a.account_id = t.account_id
    GROUP BY a.client_id
)
SELECT
    c.client_id,
    c.name,
    c.age,
    COALESCE(ca.total_accounts, 0) AS total_accounts,
    COALESCE(ca.total_balance, 0) AS total_balance,
    COALESCE(ct.total_deposits, 0) AS total_deposits,
    COALESCE(ct.total_withdrawals, 0) AS total_withdrawals
FROM clients c
LEFT JOIN client_accounts ca ON c.client_id = ca.client_id
LEFT JOIN client_transactions ct ON c.client_id = ct.client_id
WHERE c.registration_date >= '2020-01-01'
ORDER BY total_balance DESC;
'''

In [13]:
df_original = pd.read_sql_query(original_query, conn)
df_original

,client_id,name,age,total_accounts,total_balance,total_deposits,total_withdrawals
0,3,Алексей Сидоров,40,1,30000,1,1
1,5,Дмитрий Смирнов,28,1,25000,1,1
2,2,Мария Петрова,25,1,20000,1,1
3,8,Анна Морозова,29,1,18000,1,1
4,10,Татьяна Павлова,31,1,15000,0,0
5,7,Сергей Козлов,33,1,12000,1,1
6,4,Елена Кузнецова,35,1,10000,1,1


In [14]:
df_optimized = pd.read_sql_query(optimized_query, conn)
df_optimized

,client_id,name,age,total_accounts,total_balance,total_deposits,total_withdrawals
0,3,Алексей Сидоров,40,1,30000,1,1
1,5,Дмитрий Смирнов,28,1,25000,1,1
2,2,Мария Петрова,25,1,20000,1,1
3,8,Анна Морозова,29,1,18000,1,1
4,10,Татьяна Павлова,31,1,15000,0,0
5,7,Сергей Козлов,33,1,12000,1,1
6,4,Елена Кузнецова,35,1,10000,1,1


In [15]:
pd.testing.assert_frame_equal(df_original, df_optimized, check_dtype=False, check_index_type=False)

In [16]:
conn.close()